In [7]:
# Installing CUDA C++ plugin in Colab:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [8]:
import subprocess

gpu_information=subprocess.getoutput(
    "nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader,nounits"
)

if "not found" in gpu_information.lower() or "failed" in gpu_information.lower():
    raise RuntimeError("No NVIDIA GPU is available. Enable GPU in Colab runtime settings.")

gpu_name,compute_cap=map(str.strip,gpu_information.split(","))
gpu_arch=f"sm_{compute_cap.replace('.','')}"

print(f"GPU Name: {gpu_name}")
print(f"NVIDIA Architecture: {gpu_arch}")

GPU Name: Tesla T4
NVIDIA Architecture: sm_75


In [9]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [10]:
!nvidia-smi

Fri Sep 25 09:02:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
%%cuda -c "--gpu-architecture $gpu_arch"
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

__global__ void vecAddKernel(float* d_A,float* d_B,float* d_C,int N)
{
    int idx=threadIdx.x+blockIdx.x*blockDim.x;
    if(idx<N) d_C[idx]=d_A[idx]+d_B[idx];
}

void vecAdd(float* h_A,float* h_B,float* h_C,int N)
{
    for(int i=0;i<N;i++) h_C[i]=h_A[i]+h_B[i];
}

int main()
{
    int N=10;
    float *h_A,*h_B,*h_C,*h_C_CPU;

    h_A=(float*)malloc(N*sizeof(float));
    h_B=(float*)malloc(N*sizeof(float));
    h_C=(float*)malloc(N*sizeof(float));
    h_C_CPU=(float*)malloc(N*sizeof(float));

    for(int i=0;i<N;i++)
    {
        h_A[i]=i;
        h_B[i]=i;
        h_C[i]=0;
        h_C_CPU[i]=0;
    }

    vecAdd(h_A,h_B,h_C_CPU,N);

    cudaError_t error;
    float *d_A,*d_B,*d_C;
    int size=N*sizeof(float);

    error=cudaMalloc((void**)&d_A,size);
    error=cudaMemcpy(d_A,h_A,size,cudaMemcpyHostToDevice);

    error=cudaMalloc((void**)&d_B,size);
    error=cudaMemcpy(d_B,h_B,size,cudaMemcpyHostToDevice);

    error=cudaMalloc((void**)&d_C,size);

    int blockSize=256;
    int numBlocks=(N+blockSize-1)/blockSize;

    vecAddKernel<<<numBlocks,blockSize>>>(d_A,d_B,d_C,N);

    cudaDeviceSynchronize();

    error=cudaMemcpy(h_C,d_C,size,cudaMemcpyDeviceToHost);

    for(int i=0;i<N;i++)
    {
        printf("CPU: %.1f GPU: %.1f\n",h_C_CPU[i],h_C[i]);
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);
    free(h_C_CPU);

    return 0;
}

CPU: 0.0 GPU: 0.0
CPU: 2.0 GPU: 2.0
CPU: 4.0 GPU: 4.0
CPU: 6.0 GPU: 6.0
CPU: 8.0 GPU: 8.0
CPU: 10.0 GPU: 10.0
CPU: 12.0 GPU: 12.0
CPU: 14.0 GPU: 14.0
CPU: 16.0 GPU: 16.0
CPU: 18.0 GPU: 18.0



## Comparing the CPU and GPU Results:

In [14]:
%%cuda -c "--gpu-architecture $gpu_arch"
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

__global__ void vecAddKernel(const float* A,const float* B,float* C,int N)
{
    int i=blockIdx.x*blockDim.x+threadIdx.x;

    if(i<N)
        C[i]=A[i]+B[i];
}

void vecAddCPU(const float* A,const float* B,float* C,int N)
{
    for(int i=0;i<N;i++)
        C[i]=A[i]+B[i];
}

int main()
{
    const int N=10;
    const int size=N*sizeof(float);

    float *h_A=(float*)malloc(size);
    float *h_B=(float*)malloc(size);
    float *h_C_CPU=(float*)malloc(size);
    float *h_C_GPU=(float*)malloc(size);

    for(int i=0;i<N;i++)
    {
        h_A[i]=(float)i;
        h_B[i]=(float)i;
    }

    vecAddCPU(h_A,h_B,h_C_CPU,N);

    float *d_A,*d_B,*d_C;

    cudaMalloc(&d_A,size);
    cudaMalloc(&d_B,size);
    cudaMalloc(&d_C,size);

    cudaMemcpy(d_A,h_A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,h_B,size,cudaMemcpyHostToDevice);

    const int threadsPerBlock=256;
    const int blocksPerGrid=(N+threadsPerBlock-1)/threadsPerBlock;

    vecAddKernel<<<blocksPerGrid,threadsPerBlock>>>(d_A,d_B,d_C,N);

    cudaMemcpy(h_C_GPU,d_C,size,cudaMemcpyDeviceToHost);

    bool Match=true;
    for(int i=0;i<N;i++)
    {
        if(fabs(h_C_CPU[i]-h_C_GPU[i])>1e-5)
            Match=false;

        // printf("%d: %.1f + %.1f = %.1f\n", i,h_A[i],h_B[i],h_C_GPU[i]);
    }

    printf("\nResults are: %s\n",Match?"Correct":"Incorrect");

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C_CPU);
    free(h_C_GPU);

    return 0;
}


Results are: Correct

